In [ ]:
# ── Colab Setup (run this first) ──
!pip install pandas tabulate -q

import os
# Option 1: Paste key directly
os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

Notebook 03 — Data Quality Report (The Deliverable)
AutoDataEngineer: Before/After Analysis + Quality Metrics

WHAT THIS DOES:
- Compares raw vs cleaned data across every dimension
- Generates a stakeholder-ready quality report
- Produces the "receipts" that prove the system works

COLAB SETUP:
import os; os.environ["OPENAI_API_KEY"] = "sk-..."

PREREQUISITE: Run Notebooks 01 and 02 first

### Setup

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

# Load both datasets
df_raw = pd.read_csv(DATA_DIR / "raw_data.csv")
df_cleaned = pd.read_csv(DATA_DIR / "cleaned_data.csv")

# Replace empty strings with NaN for fair comparison
df_raw = df_raw.replace({"": np.nan, " ": np.nan})

print(f"Raw data:     {df_raw.shape}")
print(f"Cleaned data: {df_cleaned.shape}")

### Column-by-column null comparison

In [ ]:
print("=" * 70)
print("NULL VALUES — BEFORE vs AFTER")
print("=" * 70)

comparison_rows = []
for col in df_raw.columns:
    before = int(df_raw[col].isnull().sum())
    after = int(df_cleaned[col].isnull().sum()) if col in df_cleaned.columns else "DROPPED"
    if isinstance(after, int):
        fixed = before - after
        pct_fixed = f"{(fixed / before * 100):.0f}%" if before > 0 else "N/A"
    else:
        fixed = "—"
        pct_fixed = "—"
    comparison_rows.append({
        "Column": col,
        "Nulls Before": before,
        "Nulls After": after,
        "Fixed": fixed,
        "% Fixed": pct_fixed,
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

total_before = df_raw.isnull().sum().sum()
total_after = df_cleaned.isnull().sum().sum()
print(f"\nTotal: {total_before} nulls → {total_after} nulls ({total_before - total_after} fixed)")

### Duplicate analysis

In [ ]:
print("\n" + "=" * 70)
print("DUPLICATES — BEFORE vs AFTER")
print("=" * 70)

dup_before = df_raw.duplicated().sum()
dup_after = df_cleaned.duplicated().sum()
print(f"  Duplicate rows before: {dup_before}")
print(f"  Duplicate rows after:  {dup_after}")
print(f"  Removed: {dup_before - dup_after}")

### Data type consistency check

In [ ]:
print("\n" + "=" * 70)
print("DATA TYPES — BEFORE vs AFTER")
print("=" * 70)

for col in df_cleaned.columns:
    before_type = str(df_raw[col].dtype) if col in df_raw.columns else "N/A"
    after_type = str(df_cleaned[col].dtype)
    changed = " ← FIXED" if before_type != after_type else ""
    print(f"  {col:30s} {before_type:15s} → {after_type:15s}{changed}")

### Value consistency check

In [ ]:
print("\n" + "=" * 70)
print("VALUE CONSISTENCY — SAMPLE CHECKS")
print("=" * 70)

# Check a few columns for standardization
for col in ["nutrition_grade_fr", "countries", "brands", "categories", "packaging"]:
    if col in df_cleaned.columns:
        unique_before = df_raw[col].dropna().nunique() if col in df_raw.columns else 0
        unique_after = df_cleaned[col].dropna().nunique()
        print(f"\n  {col}:")
        print(f"    Unique values: {unique_before} → {unique_after}")
        if unique_after <= 15:
            print(f"    Clean values:  {sorted(df_cleaned[col].dropna().unique().tolist())}")

### Numeric column validation

In [ ]:
print("\n" + "=" * 70)
print("NUMERIC COLUMNS — VALIDATION")
print("=" * 70)

for col in ["energy_100g", "fat_100g", "sugars_100g", "proteins_100g"]:
    if col in df_cleaned.columns:
        print(f"\n  {col}:")
        # Try to convert to numeric
        numeric_vals = pd.to_numeric(df_cleaned[col], errors="coerce")
        non_numeric = numeric_vals.isnull().sum() - df_cleaned[col].isnull().sum()
        print(f"    Non-numeric values remaining: {max(0, non_numeric)}")
        valid = numeric_vals.dropna()
        if len(valid) > 0:
            print(f"    Range: {valid.min():.1f} — {valid.max():.1f}")
            print(f"    Mean:  {valid.mean():.1f}")
            negatives = (valid < 0).sum()
            if negatives > 0:
                print(f"    WARNING: {negatives} negative values")

### Generate the full quality report

In [ ]:
print("\n" + "=" * 70)
print("FINAL DATA QUALITY REPORT")
print("=" * 70)

report = {
    "dataset": {
        "rows_before": len(df_raw),
        "rows_after": len(df_cleaned),
        "columns_before": len(df_raw.columns),
        "columns_after": len(df_cleaned.columns),
    },
    "nulls": {
        "total_before": int(total_before),
        "total_after": int(total_after),
        "fixed": int(total_before - total_after),
        "fix_rate": f"{(total_before - total_after) / max(total_before, 1) * 100:.1f}%",
    },
    "duplicates": {
        "before": int(dup_before),
        "after": int(dup_after),
        "removed": int(dup_before - dup_after),
    },
    "quality_score": {},
}

# Calculate per-column quality score
scores = []
for col in df_cleaned.columns:
    col_score = 100
    null_pct = df_cleaned[col].isnull().mean() * 100
    col_score -= null_pct  # Deduct for nulls
    scores.append({"column": col, "score": round(max(0, col_score), 1)})

avg_score = sum(s["score"] for s in scores) / len(scores)
report["quality_score"] = {
    "overall": round(avg_score, 1),
    "per_column": scores,
}

print(f"""
  Rows:              {report['dataset']['rows_before']} → {report['dataset']['rows_after']}
  Nulls fixed:       {report['nulls']['fixed']} ({report['nulls']['fix_rate']})
  Duplicates removed:{report['duplicates']['removed']}
  Quality score:     {report['quality_score']['overall']}/100

  Per-column scores:""")

for s in scores:
    bar = "█" * int(s["score"] / 5)
    print(f"    {s['column']:30s} {s['score']:5.1f}/100 {bar}")

### Generate Markdown report

In [ ]:
md = "# AutoDataEngineer — Data Quality Report\n\n"
md += "---\n\n"
md += "## Overview\n\n"
md += f"| Metric | Before | After | Change |\n"
md += f"|--------|--------|-------|--------|\n"
md += f"| Rows | {report['dataset']['rows_before']} | {report['dataset']['rows_after']} | {report['dataset']['rows_before'] - report['dataset']['rows_after']} removed |\n"
md += f"| Null values | {report['nulls']['total_before']} | {report['nulls']['total_after']} | {report['nulls']['fixed']} fixed |\n"
md += f"| Duplicate rows | {report['duplicates']['before']} | {report['duplicates']['after']} | {report['duplicates']['removed']} removed |\n"
md += f"| Quality score | — | {report['quality_score']['overall']}/100 | — |\n\n"

md += "## Column Quality Scores\n\n"
md += "| Column | Score | Status |\n"
md += "|--------|-------|--------|\n"
for s in scores:
    status = "Excellent" if s["score"] >= 90 else ("Good" if s["score"] >= 70 else "Needs Work")
    md += f"| {s['column']} | {s['score']}/100 | {status} |\n"

md += f"\n---\n\n*Generated by AutoDataEngineer — autonomous multi-agent data cleaning system*\n"

# Display
try:
    from IPython.display import display, Markdown
    display(Markdown(md))
except ImportError:
    print(md)

# Save report
with open(DATA_DIR / "quality_report.json", "w") as f:
    json.dump(report, f, indent=2)

with open(DATA_DIR / "quality_report.md", "w") as f:
    f.write(md)

print(f"\nSaved: quality_report.json + quality_report.md")
print("Notebook 03 complete!")